In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import os

# --- 1. Load the Dataset ---
csv_path = os.path.join("dataset", "keypoint.csv")
model_dir = os.path.join("model", "keypoint_classifier")
model_path = os.path.join(model_dir, "gesture_classifier.pkl")

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Could not find your dataset at {csv_path}. Make sure you collected data first!")

# Read the CSV into a Pandas DataFrame
df = pd.read_csv(csv_path)
print(f" Successfully loaded dataset. Total samples: {len(df)}")

# --- 2. Split Features and Labels ---
# X contains the 42 coordinate values (inputs), Y contains the gesture ID label (output)
X = df.drop(columns=['label']).values
y = df['label'].values

# Split data into 80% training and 20% testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f" Training samples: {len(X_train)} | Testing samples: {len(X_test)}")

# --- 3. Define and Train the MLP Neural Network ---
print("\n Training the lightweight Keypoint Classifier...")

# We use a small architecture: 2 hidden layers (hidden units: 32, 16)
# This keeps the model size tiny (< 50KB) and extremely fast for live webcam loops.
clf = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42
)

clf.fit(X_train, y_train)
print(" Model training complete!")

# --- 4. Evaluate Model Performance ---
print("\n --- EVALUATION REPORT ---")
y_pred = clf.predict(X_test)

# Print overall accuracy and precision matrix
print(classification_report(y_test, y_pred))

print("--- CONFUSION MATRIX ---")
print(confusion_matrix(y_test, y_pred))

# --- 5. Save the Trained Model Weights ---
os.makedirs(model_dir, exist_ok=True)
joblib.dump(clf, model_path)
print(f"\n Model weights saved securely at: {model_path}")

 Successfully loaded dataset. Total samples: 4543
 Training samples: 3634 | Testing samples: 909

 Training the lightweight Keypoint Classifier...
 Model training complete!

 --- EVALUATION REPORT ---
              precision    recall  f1-score   support

           0       0.99      0.99      0.99        91
           1       1.00      0.99      1.00       146
           2       1.00      1.00      1.00       133
           3       0.99      0.99      0.99       200
           4       1.00      1.00      1.00       339

    accuracy                           1.00       909
   macro avg       1.00      1.00      1.00       909
weighted avg       1.00      1.00      1.00       909

--- CONFUSION MATRIX ---
[[ 90   0   0   1   0]
 [  0 145   0   1   0]
 [  0   0 133   0   0]
 [  1   0   0 199   0]
 [  0   0   0   0 339]]

 Model weights saved securely at: model\keypoint_classifier\gesture_classifier.pkl
